In [1]:
import json
import sys
import os
# Temporarily add MedRAG repo
sys.path.append(os.path.abspath("../MedRAG"))
from src.medrag import MedRAG
from tqdm import tqdm
import re

/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [44]:
!ls ../data/ophthalmology_test_sets/nei_qa.json

../data/ophthalmology_test_sets/nei_qa.json


In [59]:
def standardize_question(text):
    pattern = r"Question: (.*?)([a]\).*?)([b]\).*?)([c]\).*?)([d]\).*)"

    match = re.search(pattern, text, re.DOTALL)
    if match:
        question = match.group(1)
        choices = {
            "A": match.group(2).strip()[3:],  # Remove 'A. ' from the beginning
            "B": match.group(3).strip()[3:],  # Remove 'B. ' from the beginning
            "C": match.group(4).strip()[3:],  # Remove 'C. ' from the beginning
            "D": match.group(5).strip()[3:]   # Remove 'D. ' from the beginning
        }
        return question, choices
    else:
        print("[ERROR]: ", text)


with open('../data/ophthalmology_test_sets/multi_ophtha.json', 'r') as file:
    data = json.load(file)

standardize_question(data[50]['query'])

('A 60-year-old diabetic patient complains of pain and low visual acuity in the right eye. On ophthalmological examination, visual acuity was 0.1, intraocular pressure 45 mmHg, corneal edema, iris neovascularization, open angle in all quadrants on gonioscopy, cup/disc ratio 0.3, and proliferative diabetic retinopathy. After initiating hypotensive clinical treatment, the patient had a good response. Among the options below, which is the most appropriate course of action to follow? \n',
 {'A': 'Cyclophotocoagulation.',
  'B': 'Drainage implant.',
  'C': 'Panretinal photocoagulation.',
  'D': 'Trabeculectomy with mitomycin C.'})

In [62]:
for x in data:
    standardize_question(x['query'])

In [36]:
def standardize_question(text):
    pattern = r"Question: (.*?)\n([a]\).*?)(\n[b]\).*?)(\n[c]\).*?)(\n[d]\).*)"

    match = re.search(pattern, text, re.DOTALL)
    if match:
        question = match.group(1)
        choices = {
            "A": match.group(2).strip()[3:],  # Remove 'A. ' from the beginning
            "B": match.group(3).strip()[3:],  # Remove 'B. ' from the beginning
            "C": match.group(4).strip()[3:],  # Remove 'C. ' from the beginning
            "D": match.group(5).strip()[3:]   # Remove 'D. ' from the beginning
        }
        return question, choices
    else:
        print("[ERROR]: ", text)


with open('../data/ophthalmology_test_sets/multi_ophtha.json', 'r') as file:
    data = json.load(file)

standardize_question(data[1]['query'])

('Mark the alternative that best correlates the histological characteristics with the respective ocular tissues:\n\nI. Monolayer of cells tightly joined together by junctional complexes.\nII. Parallel and regular striations observed under optical microscopy, perpendicular to the epithelium.\nIII. It contains bipolar cells, amacrine cells, horizontal cells and Muller cells.\nIV. It contains magnocellular, parvocellular and coniocellular cells.\n\nA. Photoreceptors.\nB. Retinal pigmented epithelium.\nC. Retinal ganglionic layer.\nD. Inner nuclear layer.\n',
 {'A': 'I: A, II: C; III: D; IV: B.',
  'B': 'I: B; II:A; III: D; IV: C.',
  'C': 'I: B; II: A; III: C; IV: C.',
  'D': 'I: C; II: A; III: D; IV: B.'})

In [26]:
print(data[3]['query'])

Given a multiple choice question in the field of ophthalmology, select the correct answer from the given options.

Question: Regarding Descemet's membrane of the cornea, it is correct to state:
a) Endothelial cells do not participate in its formation.
b) Its thickness in the adult is about 30 µm.
c) Its most anterior portion is of embryonic origin.
d) Its thickness reduces with age.


In [18]:
data[1]

{'id': '2',
 'query': 'Given a multiple choice question in the field of ophthalmology, select the correct answer from the given options.\n\nQuestion: Mark the alternative that best correlates the histological characteristics with the respective ocular tissues:\n\nI. Monolayer of cells tightly joined together by junctional complexes.\nII. Parallel and regular striations observed under optical microscopy, perpendicular to the epithelium.\nIII. It contains bipolar cells, amacrine cells, horizontal cells and Muller cells.\nIV. It contains magnocellular, parvocellular and coniocellular cells.\n\nA. Photoreceptors.\nB. Retinal pigmented epithelium.\nC. Retinal ganglionic layer.\nD. Inner nuclear layer.\n\na) I: A, II: C; III: D; IV: B.\nb) I: B; II:A; III: D; IV: C.\nc) I: B; II: A; III: C; IV: C.\nd) I: C; II: A; III: D; IV: B.',
 'answer': 'B'}

In [2]:
data_train = []
with open("../MedMCQA/train.json", "r") as file:
    for line in file:
        question = json.loads(line)
        data_train.append(question)

In [3]:
data_validation = []
with open("../MedMCQA/dev.json", "r") as file:
    for line in file:
        question = json.loads(line)
        data_validation.append(question)

In [4]:
len(data_validation)

4183

In [5]:
data_train_top1k = data_train[:1000]
data_validation_top1k = data_validation[:1000]

In [ ]:
def get_answer(cot, q):
    answer, _, _ = cot.answer(question=q['question'], options={"A": q['opa'], "B": q['opb'], "C": q['opc'], "D": q['opd']})
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        list_of_string = re.findall(r'"(.*?)"', answer)
        if len(list_of_string) != 0:
            return list_of_string[-1]
        else:
            print('Output is not in json format!')
            print(answer)
            return None


cot = MedRAG(llm_name="OpenAI/gpt-3.5-turbo-16k", rag=False)
correct, incorrect = 0, 0
unknown = 0
for question in tqdm(data_train_top1k):
    a = get_answer(cot, question)
    mapper = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
    try:
        ans = (mapper[a[0]] == question['cop']) # True for correct answer, False otherwise
        if ans:
            correct += 1
        else:
            incorrect += 1
    except:
        unknown += 1
        print(question)
        print(a)

### Misc part

In [3]:
!ls ../data/ophthalmology_test_sets

BCSC.json					MedMCQA_ophthalmology_val.json
jama_ophthalmology_clinical_challenge_378.json


In [4]:
import json


In [1]:
import json
import sys
import os
# Temporarily add MedRAG repo
sys.path.append(os.path.abspath("../MedRAG"))
from src.medrag import MedRAG
from tqdm import tqdm
import re
import faiss
import json
import torch
import tqdm
import numpy as np

print("--------------------- Start building corpus! ---------------------")

def construct_index(index_dir, model_name, h_dim=768, HNSW=False, M=32):

    with open(os.path.join(index_dir, "metadatas.jsonl"), 'w') as f:
        f.write("")
    
    if HNSW:
        M = M
        if "specter" in model_name.lower():
            index = faiss.IndexHNSWFlat(h_dim, M)
        else:
            index = faiss.IndexHNSWFlat(h_dim, M)
            index.metric_type = faiss.METRIC_INNER_PRODUCT
    else:
        if "specter" in model_name.lower():
            index = faiss.IndexFlatL2(h_dim)
        else:
            index = faiss.IndexFlatIP(h_dim)

    for fname in tqdm.tqdm(sorted(os.listdir(os.path.join(index_dir, "embedding")))):
        print(os.path.join(index_dir, "embedding", fname))
        curr_embed = np.load(os.path.join(index_dir, "embedding", fname))
        index.add(curr_embed)
        with open(os.path.join(index_dir, "metadatas.jsonl"), 'a+') as f:
            f.write("\n".join([json.dumps({'index': i, 'source': fname.replace(".npy", "")}) for i in range(len(curr_embed))]) + '\n')

    faiss.write_index(index, os.path.join(index_dir, "faiss.index"))
    return index

class Generate_embed: 

    def __init__(self, retriever_name="ncbi/MedCPT-Query-Encoder", corpus_name="textbooks", db_dir="./corpus", HNSW=False, **kwarg):
        self.corpus_name = "wikipedia"
        self.retriever_name = "ncbi/MedCPT-Query-Encoder"
        self.index_dir = "./corpus/wikipedia/index/ncbi/MedCPT-Article-Encoder"

        if self.corpus_name in ["textbooks", "pubmed", "wikipedia"] and self.retriever_name in ["allenai/specter", "facebook/contriever", "ncbi/MedCPT-Query-Encoder"] and not os.path.exists(os.path.join(self.index_dir, "embedding")):
            print("[In progress] Downloading the {:s} embeddings given by the {:s} model...".format(self.corpus_name, self.retriever_name.replace("Query-Encoder", "Article-Encoder")))
            os.makedirs(self.index_dir, exist_ok=True)
            if self.corpus_name == "textbooks":
                if self.retriever_name == "allenai/specter":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EYRRpJbNDyBOmfzCOqfQzrsBwUX0_UT8-j_geDPcVXFnig?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "facebook/contriever":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EQqzldVMCCVIpiFV4goC7qEBSkl8kj5lQHtNq8DvHJdAfw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EQ8uXe4RiqJJm0Tmnx7fUUkBKKvTwhu9AqecPA3ULUxUqQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
            elif self.corpus_name == "pubmed":
                if self.retriever_name == "allenai/specter":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/Ebz8ySXt815FotxC1KkDbuABNycudBCoirTWkKfl8SEswA?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "facebook/contriever":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EWecRNfTxbRMnM0ByGMdiAsBJbGJOX_bpnUoyXY9Bj4_jQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EVCuryzOqy5Am5xzRu6KJz4B6dho7Tv7OuTeHSh3zyrOAw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
            elif self.corpus_name == "wikipedia":
                if self.retriever_name == "allenai/specter":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/Ed7zG3_ce-JOmGTbgof3IK0BdD40XcuZ7AGZRcV_5D2jkA?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "facebook/contriever":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/ETKHGV9_KNBPmDM60MWjEdsBXR4P4c7zZk1HLLc0KVaTJw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EXoxEANb_xBFm6fa2VLRmAcBIfCuTL-5VH6vl4GxJ06oCQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
            os.system("unzip {:s} -d {:s}".format(os.path.join(self.index_dir, "embedding.zip"), self.index_dir))
            os.system("rm {:s}".format(os.path.join(self.index_dir, "embedding.zip")))
            h_dim = 768
        else:
            h_dim = 768
            print("No need to generate embedding! Proceed to index.")
            # h_dim = embed(chunk_dir=self.chunk_dir, index_dir=self.index_dir, model_name=self.retriever_name.replace("Query-Encoder", "Article-Encoder"), **kwarg)
    
        print("[In progress] Embedding finished! The dimension of the embeddings is {:d}.".format(h_dim))
        self.index = construct_index(index_dir=self.index_dir, model_name=self.retriever_name.replace("Query-Encoder", "Article-Encoder"), h_dim=h_dim, HNSW=True)
        print("[Finished] Corpus indexing finished!")
        self.metadatas = [json.loads(line) for line in open(os.path.join(self.index_dir, "metadatas.jsonl")).read().strip().split('\n')]            


Generate_embed()


"""
model_name_ = "OpenAI/gpt-35-turbo-16k"
retriever_name_ = "MedCPT"
corpus_name_ = "Wikipedia"
# corpus_name_ = "Textbooks"
output_name_ = "MedMCQA_train_top1k+" + model_name_ + "+" + retriever_name_ + "+" + corpus_name_
cot = MedRAG(llm_name=model_name_, rag=True, retriever_name=retriever_name_, corpus_name=corpus_name_)
"""

/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--------------------- Start building corpus! ---------------------
No need to generate embedding! Proceed to index.
[In progress] Embedding finished! The dimension of the embeddings is 768.


  0%|          | 0/445 [00:00<?, ?it/s]

./corpus/wikipedia/index/ncbi/MedCPT-Article-Encoder/embedding/wiki20220301en000.npy


  0%|          | 1/445 [11:31<85:17:13, 691.52s/it]

./corpus/wikipedia/index/ncbi/MedCPT-Article-Encoder/embedding/wiki20220301en001.npy


  0%|          | 2/445 [23:56<88:24:20, 718.42s/it]


./corpus/wikipedia/index/ncbi/MedCPT-Article-Encoder/embedding/wiki20220301en002.npy


ValueError: cannot reshape array of size 16777184 into shape (232622,768)

In [ ]:
import os
class Generate_embed: 

    def __init__(self, retriever_name="ncbi/MedCPT-Query-Encoder", corpus_name="textbooks", db_dir="./corpus", HNSW=False, **kwarg):
        sxelf.corpus_name = "wikipedia"
        self.retriever_name = "ncbi/MedCPT-Query-Encoder"
        self.index_dir = "./corpus/wikipedia/index/ncbi/MedCPT-Article-Encoder"

        if self.corpus_name in ["textbooks", "pubmed", "wikipedia"] and self.retriever_name in ["allenai/specter", "facebook/contriever", "ncbi/MedCPT-Query-Encoder"] and not os.path.exists(os.path.join(self.index_dir, "embedding")):
            print("[In progress] Downloading the {:s} embeddings given by the {:s} model...".format(self.corpus_name, self.retriever_name.replace("Query-Encoder", "Article-Encoder")))
            os.makedirs(self.index_dir, exist_ok=True)
            if self.corpus_name == "textbooks":
                if self.retriever_name == "allenai/specter":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EYRRpJbNDyBOmfzCOqfQzrsBwUX0_UT8-j_geDPcVXFnig?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "facebook/contriever":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EQqzldVMCCVIpiFV4goC7qEBSkl8kj5lQHtNq8DvHJdAfw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EQ8uXe4RiqJJm0Tmnx7fUUkBKKvTwhu9AqecPA3ULUxUqQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
            elif self.corpus_name == "pubmed":
                if self.retriever_name == "allenai/specter":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/Ebz8ySXt815FotxC1KkDbuABNycudBCoirTWkKfl8SEswA?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "facebook/contriever":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EWecRNfTxbRMnM0ByGMdiAsBJbGJOX_bpnUoyXY9Bj4_jQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EVCuryzOqy5Am5xzRu6KJz4B6dho7Tv7OuTeHSh3zyrOAw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
            elif self.corpus_name == "wikipedia":
                if self.retriever_name == "allenai/specter":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/Ed7zG3_ce-JOmGTbgof3IK0BdD40XcuZ7AGZRcV_5D2jkA?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "facebook/contriever":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/ETKHGV9_KNBPmDM60MWjEdsBXR4P4c7zZk1HLLc0KVaTJw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
                elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
                    os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EXoxEANb_xBFm6fa2VLRmAcBIfCuTL-5VH6vl4GxJ06oCQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
            os.system("unzip {:s} -d {:s}".format(os.path.join(self.index_dir, "embedding.zip"), self.index_dir))
            os.system("rm {:s}".format(os.path.join(self.index_dir, "embedding.zip")))
            h_dim = 768
        else:
            h_dim = embed(chunk_dir=self.chunk_dir, index_dir=self.index_dir, model_name=self.retriever_name.replace("Query-Encoder", "Article-Encoder"), **kwarg)
    
        print("[In progress] Embedding finished! The dimension of the embeddings is {:d}.".format(h_dim))
        self.index = construct_index(index_dir=self.index_dir, model_name=self.retriever_name.replace("Query-Encoder", "Article-Encoder"), h_dim=h_dim, HNSW=HNSW)
        print("[Finished] Corpus indexing finished!")
        self.metadatas = [json.loads(line) for line in open(os.path.join(self.index_dir, "metadatas.jsonl")).read().strip().split('\n')]            


Generate_embed()

[In progress] Downloading the wikipedia embeddings given by the ncbi/MedCPT-Article-Encoder model...


--2025-01-07 20:15:37--  https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EXoxEANb_xBFm6fa2VLRmAcBIfCuTL-5VH6vl4GxJ06oCQ?download=1
Resolving myuva-my.sharepoint.com (myuva-my.sharepoint.com)... 13.107.138.10, 13.107.136.10, 2620:1ec:8f8::10, ...
Connecting to myuva-my.sharepoint.com (myuva-my.sharepoint.com)|13.107.138.10|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /personal/hhu4zu_virginia_edu/Documents/MedRAG/corpus/wikipedia/index/ncbi/MedCPT-Article-Encoder/embedding.zip?ga=1 [following]
--2025-01-07 20:15:38--  https://myuva-my.sharepoint.com/personal/hhu4zu_virginia_edu/Documents/MedRAG/corpus/wikipedia/index/ncbi/MedCPT-Article-Encoder/embedding.zip?ga=1
Reusing existing connection to myuva-my.sharepoint.com:443.
HTTP request sent, awaiting response... 200 OK
Length: 85094921091 (79G) [application/x-zip-compressed]
Saving to: ‘./corpus/wikipedia/index/ncbi/MedCPT-Article-Encoder/embedding.zip’

     0K .......... .......... .

In [ ]:
corpus_name = "wikipedia"
retriever_name = "ncbi/MedCPT-Query-Encoder"

if self.corpus_name in ["textbooks", "pubmed", "wikipedia"] and self.retriever_name in ["allenai/specter", "facebook/contriever", "ncbi/MedCPT-Query-Encoder"] and not os.path.exists(os.path.join(self.index_dir, "embedding")):
    print("[In progress] Downloading the {:s} embeddings given by the {:s} model...".format(self.corpus_name, self.retriever_name.replace("Query-Encoder", "Article-Encoder")))
    os.makedirs(self.index_dir, exist_ok=True)
    if self.corpus_name == "textbooks":
        if self.retriever_name == "allenai/specter":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EYRRpJbNDyBOmfzCOqfQzrsBwUX0_UT8-j_geDPcVXFnig?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "facebook/contriever":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EQqzldVMCCVIpiFV4goC7qEBSkl8kj5lQHtNq8DvHJdAfw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EQ8uXe4RiqJJm0Tmnx7fUUkBKKvTwhu9AqecPA3ULUxUqQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
    elif self.corpus_name == "pubmed":
        if self.retriever_name == "allenai/specter":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/Ebz8ySXt815FotxC1KkDbuABNycudBCoirTWkKfl8SEswA?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "facebook/contriever":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EWecRNfTxbRMnM0ByGMdiAsBJbGJOX_bpnUoyXY9Bj4_jQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EVCuryzOqy5Am5xzRu6KJz4B6dho7Tv7OuTeHSh3zyrOAw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
    elif self.corpus_name == "wikipedia":
        if self.retriever_name == "allenai/specter":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/Ed7zG3_ce-JOmGTbgof3IK0BdD40XcuZ7AGZRcV_5D2jkA?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "facebook/contriever":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/ETKHGV9_KNBPmDM60MWjEdsBXR4P4c7zZk1HLLc0KVaTJw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EXoxEANb_xBFm6fa2VLRmAcBIfCuTL-5VH6vl4GxJ06oCQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
    os.system("unzip {:s} -d {:s}".format(os.path.join(self.index_dir, "embedding.zip"), self.index_dir))
    os.system("rm {:s}".format(os.path.join(self.index_dir, "embedding.zip")))
    h_dim = 768
else:
    h_dim = embed(chunk_dir=self.chunk_dir, index_dir=self.index_dir, model_name=self.retriever_name.replace("Query-Encoder", "Article-Encoder"), **kwarg)

print("[In progress] Embedding finished! The dimension of the embeddings is {:d}.".format(h_dim))
self.index = construct_index(index_dir=self.index_dir, model_name=self.retriever_name.replace("Query-Encoder", "Article-Encoder"), h_dim=h_dim, HNSW=HNSW)
print("[Finished] Corpus indexing finished!")
self.metadatas = [json.loads(line) for line in open(os.path.join(self.index_dir, "metadatas.jsonl")).read().strip().split('\n')]            

In [ ]:
if self.corpus_name in ["textbooks", "pubmed", "wikipedia"] and self.retriever_name in ["allenai/specter", "facebook/contriever", "ncbi/MedCPT-Query-Encoder"] and not os.path.exists(os.path.join(self.index_dir, "embedding")):
    print("[In progress] Downloading the {:s} embeddings given by the {:s} model...".format(self.corpus_name, self.retriever_name.replace("Query-Encoder", "Article-Encoder")))
    os.makedirs(self.index_dir, exist_ok=True)
    if self.corpus_name == "textbooks":
        if self.retriever_name == "allenai/specter":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EYRRpJbNDyBOmfzCOqfQzrsBwUX0_UT8-j_geDPcVXFnig?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "facebook/contriever":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EQqzldVMCCVIpiFV4goC7qEBSkl8kj5lQHtNq8DvHJdAfw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EQ8uXe4RiqJJm0Tmnx7fUUkBKKvTwhu9AqecPA3ULUxUqQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
    elif self.corpus_name == "pubmed":
        if self.retriever_name == "allenai/specter":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/Ebz8ySXt815FotxC1KkDbuABNycudBCoirTWkKfl8SEswA?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "facebook/contriever":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EWecRNfTxbRMnM0ByGMdiAsBJbGJOX_bpnUoyXY9Bj4_jQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EVCuryzOqy5Am5xzRu6KJz4B6dho7Tv7OuTeHSh3zyrOAw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
    elif self.corpus_name == "wikipedia":
        if self.retriever_name == "allenai/specter":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/Ed7zG3_ce-JOmGTbgof3IK0BdD40XcuZ7AGZRcV_5D2jkA?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "facebook/contriever":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/ETKHGV9_KNBPmDM60MWjEdsBXR4P4c7zZk1HLLc0KVaTJw?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
        elif self.retriever_name == "ncbi/MedCPT-Query-Encoder":
            os.system("wget -O {:s} https://myuva-my.sharepoint.com/:u:/g/personal/hhu4zu_virginia_edu/EXoxEANb_xBFm6fa2VLRmAcBIfCuTL-5VH6vl4GxJ06oCQ?download=1".format(os.path.join(self.index_dir, "embedding.zip")))
    os.system("unzip {:s} -d {:s}".format(os.path.join(self.index_dir, "embedding.zip"), self.index_dir))
    os.system("rm {:s}".format(os.path.join(self.index_dir, "embedding.zip")))
    h_dim = 768
else:
    h_dim = embed(chunk_dir=self.chunk_dir, index_dir=self.index_dir, model_name=self.retriever_name.replace("Query-Encoder", "Article-Encoder"), **kwarg)

print("[In progress] Embedding finished! The dimension of the embeddings is {:d}.".format(h_dim))
self.index = construct_index(index_dir=self.index_dir, model_name=self.retriever_name.replace("Query-Encoder", "Article-Encoder"), h_dim=h_dim, HNSW=HNSW)
print("[Finished] Corpus indexing finished!")
self.metadatas = [json.loads(line) for line in open(os.path.join(self.index_dir, "metadatas.jsonl")).read().strip().split('\n')]            

In [5]:
model_name_ = "OpenAI/gpt-35-turbo-16k"
retriever_name_ = "MedCPT"
corpus_name_ = "StatPearls"
# corpus_name_ = "Textbooks"
output_name_ = "MedMCQA_train_top1k+" + model_name_ + "+" + retriever_name_ + "+" + corpus_name_
cot = MedRAG(llm_name=model_name_, rag=True, retriever_name=retriever_name_, corpus_name=corpus_name_)


No sentence-transformers model found with name /home/zc347/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.


/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [20]:
xxx = "asdfas/fga"
xxx = xxx.replace("/", "-")
xxx

'asdfas-fga'

In [ ]:
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

true_labels = [1, 2, 3, 4, 2]
predicted_labels = [0, 1, 2, 3, 1]  # Includes a label '0' not in true_labels

# Combine the lists to ensure all labels are known to the encoder
combined_labels = true_labels + predicted_labels
label_encoder = LabelEncoder()
label_encoder.fit(combined_labels)

# Transform labels
true_encoded = label_encoder.transform(true_labels)
predicted_encoded = label_encoder.transform(predicted_labels)

# Calculate macro F1 score
macro_f1 = f1_score(true_encoded, predicted_encoded, average='macro')
print("Macro F1 Score:", macro_f1)


In [7]:
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

true_labels = [1,2,3,4,2]
predicted_labels = [1,2,3,1,0]

label_encoder = LabelEncoder()
label_encoder.fit(true_labels+predicted_labels)
true_encoded = label_encoder.transform(true_labels)
predicted_encoded = label_encoder.transform(predicted_labels)
macro_f1 = f1_score(true_encoded, predicted_encoded, average='macro')
macro_f1

0.4666666666666666

In [10]:
!export PATH=/home/zc347/jdk-23.0.1/bin:$PATH

In [13]:
!~/.bashrc

/bin/bash: /home/zc347/.bashrc: Permission denied


In [9]:
!echo 'export PATH=/home/zc347/jdk-23.0.1/bin:$PATH' >> ~/.bashrc && source ~/.bashrc

In [10]:
!export PATH=/home/zc347/jdk-23.0.1/bin:$PATH && which java

~/jdk-23.0.1/bin/java


In [11]:
!which java

/usr/bin/which: no java in (/vast/palmer/apps/services/ood/share/software/julia-1.9.2/bin:/vast/palmer/apps/services/ood/mccleary/var_www_ood_apps/conda/ycrc_default/bin:/vast/palmer/apps/avx2/software/miniconda/24.9.2/condabin:/vast/palmer/apps/avx2/software/miniconda/24.9.2:/vast/palmer/apps/avx2/software/miniconda/24.9.2/sbin:/vast/palmer/apps/avx2/software/miniconda/24.9.2/bin:/opt/lenovo/onecli:/vast/palmer/apps/bin:/home/zc347/.local/bin:/home/zc347/bin:/opt/slurm/current/bin:/usr/local/bin:/usr/bin:/usr/local/sbin:/usr/sbin)


In [12]:
from pyserini.search.lucene import LuceneSearcher

lucene_bm25_searcher = LuceneSearcher.from_prebuilt_index('msmarco-v1-passage')
hits = lucene_bm25_searcher.search('what is a lobster roll?')

for i in range(0, 10):
    print(f'{i+1:2} {hits[i].docid:7} {hits[i].score:.5f}')

Exception: Unable to find javac

In [7]:
def get_answer(cot, q):
    answer, _, _ = cot.answer(question=q['question'], options={"A": q['opa'], "B": q['opb'], "C": q['opc'], "D": q['opd']})
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        list_of_string = re.findall(r'"(.*?)"', answer)
        if len(list_of_string) != 0:
            return list_of_string[-1]
        else:
            print('Output is not in json format!')
            print(answer)
            return None


cot = MedRAG(llm_name="OpenAI/gpt-35-turbo", rag=True, retriever_name="BM25", corpus_name="Textbooks")
correct, incorrect = 0, 0
unknown = 0
for question in tqdm([data_train_top1k[0]]):
    a = get_answer(cot, question)
    mapper = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
    try:
        ans = (mapper[a[0]] == question['cop']) # True for correct answer, False otherwise
        if ans:
            correct += 1
        else:
            incorrect += 1
    except:
        unknown += 1
        print(question)
        print(a)

Exception: Unable to find javac

In [10]:
from pyserini.search.lucene import LuceneSearcher

lucene_bm25_searcher = LuceneSearcher.from_prebuilt_index('msmarco-v1-passage')
hits = lucene_bm25_searcher.search('what is a lobster roll?')

for i in range(0, 10):
    print(f'{i+1:2} {hits[i].docid:7} {hits[i].score:.5f}')

Exception: Unable to find javac

In [6]:
# https://learn.microsoft.com/en-us/azure/ai-services/openai/quickstart?tabs=command-line%2Cpython-new&pivots=programming-language-python

# Ensure you have set the following environment variables for security and accessibility:
# export AZURE_OPENAI_API_KEY="Your-API-KEY"
# export AZURE_OPENAI_ENDPOINT="Your-ENDPOINT"

import os
from openai import AzureOpenAI

api_version = "2024-08-01-preview" # Use the correct version of the API

# Initialize the Azure OpenAI client using environment variables
client = AzureOpenAI(
    api_key="ef6e0c5e4cd949d4badab3a3492d805b",
    api_version=api_version,
    azure_endpoint="https://image-text-medical.openai.azure.com/"
)

# Define the input content
input_content = "A 34-year-old woman in her 34th week of pregnancy develops pre-eclampsia. She is monitored successfully until she presents to the emergency department with acute-onset headache, altered mental status, and complete vision loss in both eyes. Magnetic resonance imaging (MRI) shows increased fluid-attenuated inversion recovery (FLAIR) signal in the bilateral occipital lobes. After she is stabilized, how should the ophthalmologist counsel her family regarding her visual prognosis?"

# Generate a response using the Azure OpenAI client
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are an expert assistant specialized in ophthalmology."},
        {"role": "user", "content": input_content}
    ],
    temperature=0  # Set temperature to 0 for deterministic responses
)

# Output the generated response
print(response.choices[0].message.content)

The clinical presentation and MRI findings suggest that the patient has developed posterior reversible encephalopathy syndrome (PRES), a condition often associated with pre-eclampsia. PRES is characterized by symptoms such as headache, altered mental status, seizures, and visual disturbances, and it typically shows increased FLAIR signal in the posterior regions of the brain, particularly the occipital lobes.

In terms of visual prognosis, the ophthalmologist should explain the following to the patient's family:

1. **Nature of PRES**: PRES is a condition that can cause temporary changes in brain function due to swelling in the brain's posterior regions, including the occipital lobes, which are responsible for vision.

2. **Reversibility**: The term "reversible" in PRES indicates that with appropriate treatment and management of the underlying cause (in this case, pre-eclampsia), the condition often improves. Many patients experience significant recovery of their neurological and visua

In [27]:
def get_answer(cot, q):
    answer, _, _ = cot.answer(question=q['question'], options={"A": q['opa'], "B": q['opb'], "C": q['opc'], "D": q['opd']})
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        if type(answer) == dict:
            return answer['answer_choice']
        print('Output is not in json format!')
        print(answer)
        return None


cot = MedRAG(llm_name="OpenAI/gpt-3.5-turbo-16k", rag=False)
correct, incorrect = 0, 0
unknown = 0
for question in tqdm(data_validation_top1k):
    a = get_answer(cot, question)
    mapper = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
    try:
        ans = (mapper[a[0]] == question['cop']) # True for correct answer, False otherwise
        if ans:
            correct += 1
        else:
            incorrect += 1
    except:
        unknown += 1
        print(a)

  0%|          | 0/100 [00:01<?, ?it/s]


KeyboardInterrupt: 

In [24]:
print(correct, incorrect, unknown)

57 37 6


In [23]:
def get_answer(cot, q):
    answer, _, _ = cot.answer(question=q['question'], options={"A": q['opa'], "B": q['opb'], "C": q['opc'], "D": q['opd']})
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        list_of_string = re.findall(r'"(.*?)"', answer)
        if len(list_of_string) != 0:
            return list_of_string[-1]
        else:
            print('Output is not in json format!')
            print(answer)
            return None


data_first100 = data_train[65:80]
cot = MedRAG(llm_name="OpenAI/gpt-3.5-turbo-16k", rag=False)
correct, incorrect = 0, 0
unknown = 0
for question in tqdm(data_first100):
    a = get_answer(cot, question)
    mapper = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
    try:
        ans = (mapper[a[0]] == question['cop']) # True for correct answer, False otherwise
        if ans:
            correct += 1
        else:
            incorrect += 1
    except:
        unknown += 1
        print(question)
        print(a)

 20%|██        | 3/15 [00:11<00:45,  3.78s/it]


KeyboardInterrupt: 